# Model evaluation — baseline performance floor

Task [#7](https://github.com/Break-Through-Tech/AI-Governance-1C-governed-prompt-gateway/issues/7).
This evaluates the two baseline models task #6 selected on the **validation**
split against the **held-out test set**, for the first time, and records the
result as the project's baseline performance floor.

Prerequisites: run `scripts/split_and_vectorize.py` (task #5) and
`scripts/train_baseline_models.py` (task #6) first, so `data/processed/tfidf/`
and `data/processed/models/` exist. The narrative write-up is in
[`docs/baseline_eval.md`](../docs/baseline_eval.md).

In [ ]:
from pathlib import Path
import importlib.util

ROOT = Path.cwd()
if not (ROOT / 'scripts/evaluate_baseline.py').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'scripts/evaluate_baseline.py').is_file():
    raise RuntimeError('Open this notebook from the repository root or notebooks directory.')

# Loaded by file path rather than `from scripts.evaluate_baseline import ...`:
# some environments ship their own package named `scripts`, which would
# otherwise shadow this repository's scripts/ folder. (Same guard as notebook 04.)
spec = importlib.util.spec_from_file_location(
    'evaluate_baseline', ROOT / 'scripts/evaluate_baseline.py')
evaluate_baseline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(evaluate_baseline)
evaluate_baselines = evaluate_baseline.evaluate_baselines
format_confusion_matrix = evaluate_baseline.format_confusion_matrix
LABELS = evaluate_baseline.LABELS

## Evaluate both candidates on the test set

Loads `best_model.pkl` (untuned) and `best_model_tuned.pkl` (tuned) from task
#6 and measures each on `X_test`/`y_test`. The test set is read only here, and
only to measure — never to choose or tune a model.

In [ ]:
data_dir = ROOT / 'data/processed/tfidf'
models_dir = ROOT / 'data/processed/models'
output_dir = ROOT / 'data/processed/eval'
report = evaluate_baselines(data_dir, models_dir, output_dir)

print('Test support:', report['test_support'])
print('Candidates:', {k: v['model_name'] for k, v in report['candidates'].items()})

## Test-set comparison (accuracy, macro, weighted)

In [ ]:
header = f"{'candidate':<10} {'model':<22} {'acc':>6} {'macroP':>7} {'macroR':>7} {'macroF1':>8} {'wF1':>7}"
print(header)
print('-' * len(header))
for key, cand in report['candidates'].items():
    t = cand['test']
    m = t['macro']
    print(f"{key:<10} {cand['model_name']:<22} {t['accuracy']:>6.3f} "
          f"{m['precision']:>7.3f} {m['recall']:>7.3f} {m['f1']:>8.3f} {t['weighted']['f1']:>7.3f}")

## Per-class precision / recall / F1

The `toxic` and `jailbreak` classes are the governance-critical ones.

In [ ]:
for key, cand in report['candidates'].items():
    print(f"{key} — {cand['model_name']}")
    for label in LABELS:
        p = cand['test']['per_label'][label]
        print(f"    {label:>9}: P={p['precision']:.3f} R={p['recall']:.3f} "
              f"F1={p['f1']:.3f} (support {p['support']})")
    print()

## Confusion matrices

Rows are the true class, columns the predicted class. The cell to watch is
**toxic/jailbreak predicted as `safe`** — an unsafe prompt let through the
gateway.

In [ ]:
for key, cand in report['candidates'].items():
    print(format_confusion_matrix(cand['test']['confusion_matrix'],
                                  title=f"{key} — {cand['model_name']}:"))
    print()

## Validation vs test, and the 80% F1 target

In [ ]:
print(f"{'candidate':<10} {'val macroF1':>12} {'test macroF1':>13}")
for key, cand in report['candidates'].items():
    print(f"{key:<10} {cand['validation']['macro']['f1']:>12.3f} "
          f"{cand['test']['macro']['f1']:>13.3f}")

best = report['best_by_test_macro_f1']
best_f1 = report['candidates'][best]['test']['macro']['f1']
target = report['success_criterion_f1']
print(f"\nBest by test macro F1: {best} ({report['candidates'][best]['model_name']}, F1={best_f1:.3f})")
print(f"{target:.0%} F1 target: {'met' if best_f1 >= target else 'NOT met'} (gap {target - best_f1:+.3f})")

## Baseline decision — deferred to the team

`best_by_test_macro_f1` is **untuned SVM** (highest macro F1), but the two
models trade off: SVM has higher precision (fewer false alarms), tuned Logistic
Regression has higher recall on unsafe prompts (fewer toxic/jailbreak slipping
through as `safe`). Which becomes *the* baseline is a policy call recorded in
[`docs/baseline_eval.md`](../docs/baseline_eval.md); `report['official_baseline']`
stays `None` until then.

Full machine-readable results are saved to
`data/processed/eval/test_evaluation.json`.

In [ ]:
print('official_baseline (null until the team decides):', report['official_baseline'])